In [15]:
import pandas as pd
import torch
from transformers import pipeline


In [16]:
# 1. Reviews file load orders reviews dataset file
df = pd.read_csv("/content/olist_order_reviews_dataset.csv")

In [17]:
# 2. Filter out reviews that have no text comments
df_reviews = df.dropna(subset=["review_comment_message"]).copy()
print(f"Total reviews with text: {len(df_reviews)}")

Total reviews with text: 40977


In [18]:
# 3. Take a quick sample of 500 reviews for initial testing
sample_df = df_reviews.head(500).copy()

In [19]:
# 4. Initialize multilingual sentiment pipeline on GPU if available
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
 # 5. Generate predictions on review comments
predictions = classifier(
    sample_df["review_comment_message"].tolist(),
    batch_size=32,
    truncation=True,
)

In [21]:
# Extract predicted star rating (e.g., '1 star' -> 1)
sample_df["predicted_stars"] = [int(p["label"][0]) for p in predictions]

# 6. Display ground truth vs model prediction
sample_df[
    ["review_score", "predicted_stars", "review_comment_message"]
].head()

,review_score,predicted_stars,review_comment_message
3,5,3,Recebi bem antes do prazo estipulado.
4,5,5,Parabéns lojas lannister adorei comprar pela I...
9,4,3,aparelho eficiente. no site a marca do aparelh...
12,4,3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
15,5,4,"Vendedor confiável, produto ok e entrega antes..."


Finding Accuracy

In [22]:
from sklearn.metrics import accuracy_score, precision_score, classification_report
# Filter out neutral reviews (3 stars) to evaluate clear positive vs negative sentiment
filtered_df =sample_df[sample_df["review_score"] != 3].copy()

#convert ratings to binary labels: 4 - 5 starts = Positive (1), 1-2 stars = Negative (0)
true_labels = filtered_df["review_score"].apply(lambda x: 1 if x >= 4 else 0)
pred_labels = filtered_df["predicted_stars"].apply(
    lambda x: 1 if x >= 4 else 0
)

# Calculate and display the overall accuracy rate
acc = accuracy_score(true_labels, pred_labels)
print(f"Sentiment Model Accuracy: {acc * 100:.2f}%\n")

# Display detailed metrics including precision, recall, and F1-score
print("Classification Report:")
print(
    classification_report(
        true_labels, pred_labels, target_names=["Negative", "Positive"]
    )
)



Sentiment Model Accuracy: 84.63%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.66      0.95      0.78       129
    Positive       0.97      0.81      0.88       320

    accuracy                           0.85       449
   macro avg       0.82      0.88      0.83       449
weighted avg       0.88      0.85      0.85       449



In [23]:
!pip install -q chromadb sentence-transformers

In [24]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load a multilingual embedding model
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Initialize a local persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db_reviews")
collection = client.get_or_create_collection("customer_reviews")

# 3. Prepare text, IDs, and metadata from the sample reviews
documents = sample_df["review_comment_message"].tolist()
ids = [str(val) for val in sample_df["review_id"].tolist()]
metadatas = (
    sample_df[["order_id", "review_score", "predicted_stars"]]
    .astype(str)
    .to_dict(orient="records")
)

# 4. Generate embeddings
print("Generating embeddings...")
embeddings = embed_model.encode(documents, show_progress_bar=True).tolist()

# 5. Add to ChromaDB collection
collection.add(
    documents=documents, embeddings=embeddings, ids=ids, metadatas=metadatas
)
print("All reviews successfully indexed in ChromaDB!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

All reviews successfully indexed in ChromaDB!


In [25]:
!zip -r chroma_db_reviews.zip chroma_db_reviews

updating: chroma_db_reviews/ (stored 0%)
updating: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/ (stored 0%)
updating: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/header.bin (deflated 59%)
updating: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/data_level0.bin (deflated 13%)
updating: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/link_lists.bin (deflated 87%)
updating: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/length.bin (deflated 56%)
updating: chroma_db_reviews/chroma.sqlite3 (deflated 34%)
  adding: chroma_db_reviews/b0ce061e-1667-4fd3-a3f6-92a2776cb3b5/index_metadata.pickle (deflated 63%)
